# Stage 6 — SNOMED Ancestors & Attribute Context

For each Stage 5 mapped concept:

1. Collect **2 levels of Is-a ancestors** (parent + grandparent)
2. Collect **outbound-only** attribute targets (no inverses — avoids drug→poisoning noise):
   - `cause_of` → **Due to** (`42752001`)
   - `has_causative_agent` (`246075003`) — outbound only
   - `has_finding_site` (`363698007`)
   - `has_associated_morphology` (`116676008`)
   - `has_pathological_process` (`370135005`)
   - `after` (`255234002`), `associated_with` (`47429007`)
   - `occurrence` (`246454002`), `clinical_course` (`263502005`)
   - plus one Is-a parent of each attribute target (route context)
3. **Score with MiniLM** (`all-MiniLM-L6-v2`, local) vs **patient IE entity term**
   - **Retain** if cosine similarity ≥ **0.70**
   - **high_confidence** if ≥ **0.80**

**Input:** `data/stage_05_snomed_mapping/snomed_mappings.json`  
**Output:** aggregate + per-admission `snomed_ancestors.json`, `snomed_retained.json` / `.txt`  
**Note:** First run after attribute-type changes needs `force_rebuild=True` on the index.  
**Deps:** `pip install sentence-transformers`

In [5]:
import json
import sys
import importlib
from pathlib import Path

NB = Path.cwd()
if not (NB / "pipeline.py").exists():
    if (NB / "notebooks" / "pipeline.py").exists():
        NB = NB / "notebooks"
    else:
        NB = NB.parent / "notebooks" if (NB.parent / "notebooks" / "pipeline.py").exists() else NB
sys.path.insert(0, str(NB))
REPO = NB.parent if NB.name == "notebooks" else NB

# Reload after snomed_ct.py changes (Jupyter caches the first import)
import snomed_ct
importlib.reload(snomed_ct)

from pipeline import EXPORT_DIR, print_pipeline_banner
from snomed_ct import (
    ATTR_TYPE_IDS,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_HIGH_CONF_COSINE_SIM,
    DEFAULT_MIN_COSINE_SIM,
    build_snomed_index,
    export_mappings_to_patient_folders,
    export_retained_to_patient_folders,
    find_snomed_root,
    run_stage06_ancestors,
    write_json,
)

print_pipeline_banner()
print("Attribute typeIds (weights deferred):")
for k, v in ATTR_TYPE_IDS.items():
    print(f"  {k:30s} → {v}")
print(f"Embedding model : {DEFAULT_EMBEDDING_MODEL}")
print(f"Anchor          : patient IE entity term")
print(f"Retain if sim   ≥ {DEFAULT_MIN_COSINE_SIM}")
print(f"High-conf if    ≥ {DEFAULT_HIGH_CONF_COSINE_SIM}")

STAGE_05 = REPO / "data" / "stage_05_snomed_mapping" / "snomed_mappings.json"
STAGE_06_DIR = REPO / "data" / "stage_06_snomed_ancestors"
STAGE_06_DIR.mkdir(parents=True, exist_ok=True)
if not STAGE_05.exists():
    raise FileNotFoundError(f"Run stage_05 first. Missing {STAGE_05}")

Pipeline mode : FULL (15 patients)
LLM provider  : OpenRouter (qwen/qwen-2.5-7b-instruct, ZDR on)
Qwen pair     : Local equivalent: ollama pull qwen2.5:7b
Admissions/patient (min): 2
Data dir      : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data
Export dir    : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
OpenRouter ZDR : enabled (provider.zdr=true on every request)
Attribute typeIds (weights deferred):
  cause_of                       → 42752001
  has_causative_agent            → 246075003
  has_finding_site               → 363698007
  has_associated_morphology      → 116676008
  has_pathological_process       → 370135005
Embedding model : sentence-transformers/all-MiniLM-L6-v2
Anchor          : patient IE entity term
Retain if sim   ≥ 0.7
High-conf if    ≥ 0.8


In [6]:
index = build_snomed_index(
    snomed_root=find_snomed_root(REPO / "data"),
    cache_path=REPO / "data" / "snomed_index" / "snomed_index.pkl",
    # Set True only after changing ATTR_TYPE_IDS in snomed_ct.py
    force_rebuild=False,
)
stage05 = json.loads(STAGE_05.read_text(encoding="utf-8"))
print(f"Stage 5 entities: {stage05['n_entities']} (mapped {stage05['n_mapped']})")

Loading SNOMED index cache → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/snomed_index/snomed_index.pkl
Stage 5 entities: 626 (mapped 509)


In [7]:
payload = run_stage06_ancestors(
    stage05,
    index,
    min_similarity=DEFAULT_MIN_COSINE_SIM,
    high_conf_similarity=DEFAULT_HIGH_CONF_COSINE_SIM,
    embedding_model=DEFAULT_EMBEDDING_MODEL,
    use_embeddings=True,
)
print(f"Method             : {payload.get('similarity_method')} ({payload.get('embedding_model')})")
print(f"Entities w/ retained: {payload.get('n_entities_with_retained')}")
print(f"Retained links      : {payload['n_retained_context_links']}")
print(f"High-conf links     : {payload.get('n_high_confidence_links')}")

# Preview first entity that has retained context
shown = False
for row in payload["results"]:
    sn = row.get("snomed") or {}
    ctx = row.get("ontology_context") or {}
    if not sn.get("mapped") or not (ctx.get("retained") or []):
        continue
    print(f"\nEntity: {row.get('term')!r} → {sn.get('preferred_term')} ({sn.get('concept_id')})")
    print(f"  anchor (IE term): {ctx.get('anchor_term')!r}")
    print(f"  is_a ancestors (all): {len(ctx.get('ancestors_depth2_all') or [])}")
    print(f"  attribute links (all): {len(ctx.get('attribute_relations_all') or [])}")
    print(f"  retained (sim ≥ {ctx.get('min_cosine_similarity')}): {len(ctx.get('retained') or [])}")
    for r in (ctx.get("retained") or [])[:8]:
        hc = " HIGH" if r.get("high_confidence") else ""
        print(
            f"    • [{r.get('relation')}] {r.get('term')} "
            f"(sim={r.get('cosine_similarity')}, dist={r.get('cosine_distance')}){hc}"
        )
    shown = True
    break
if not shown:
    print("No retained context links at this threshold.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...


Batches: 100%|██████████| 101/101 [00:00<00:00, 125.65it/s]


Method             : minilm (sentence-transformers/all-MiniLM-L6-v2)
Entities w/ retained: 185
Retained links      : 602
High-conf links     : 121

Entity: 'heavy vaginal bleeding' → Heavy episode of vaginal bleeding (315224006)
  anchor (IE term): 'heavy vaginal bleeding'
  is_a ancestors (all): 3
  attribute links (all): 5
  retained (sim ≥ 0.7): 1
    • [is_a] Bleeding from vagina (sim=0.8446, dist=0.1554) HIGH


In [8]:
out = write_json(STAGE_06_DIR / "snomed_ancestors.json", payload)
n = export_mappings_to_patient_folders(payload, EXPORT_DIR, "snomed_ancestors.json")
n_ret = export_retained_to_patient_folders(payload, EXPORT_DIR)
print(f"Saved → {out}")
print(f"Full ancestors per admission : {n} × snomed_ancestors.json")
print(f"Retained-only per admission  : {n_ret} × snomed_retained.json + snomed_retained.txt")

Saved → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_06_snomed_ancestors/snomed_ancestors.json
Full ancestors per admission : 15 × snomed_ancestors.json
Retained-only per admission  : 15 × snomed_retained.json + snomed_retained.txt
